# Компиляция модели под LinQ TPU в Google Colab

SDK LinQ (`tpu_framework`) работает только на **Linux x86_64 + Python 3.12**, поэтому компилируем здесь.

Что делает ноутбук:
1. Берёт проект `safety_monitor` (загрузить zip или с Google Диска).
2. Скачивает SDK LinQ по ссылке из инструкции организаторов.
3. Создаёт **отдельное окружение Python 3.12** и ставит туда SDK. SDK требует свои версии
   (`numpy==1.26.4`, `torch==2.9.1`, `tensorflow==2.20.0`, `onnx~=1.16`) — в основном окружении Colab они бы всё сломали.
4. Режет YOLO: голова (все свёртки) → TPU, хвост (DFL, декодирование боксов) → CPU.
5. Калибрует INT8 на **реальных кадрах** и компилирует голову в программу `.tpu`.
6. Проверяет точность INT8 прямо здесь, на CPU, без ускорителя.
7. Отдаёт архив: `*_head.tpu` + `*_head.onnx` + `*_tail.onnx`.

**Среда выполнения:** GPU не нужен, хватит обычного CPU. Установка занимает ~10 мин, компиляция — 5–20 мин.

Запускай ячейки по порядку: **Среда выполнения → Выполнить все** (Ctrl+F9).

In [ ]:
#@title 1. Параметры
ONNX = "models/ppe.onnx"               #@param {type:"string"}
CALIB_DIR = "datasets/ppe/val/images"  #@param {type:"string"}
N_CALIB = 200                          #@param {type:"integer"}
PRESET = "o1"                          #@param ["o1", "o5"]
PERCENTILE = 0                         #@param {type:"number"}
N_CHECK = 50                           #@param {type:"integer"}
USE_DRIVE = False                      #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/hackathon"  #@param {type:"string"}

# SDK LinQ — ссылка из «Процедура подготовки и инференса сетей на LinQ TPU»
FRAMEWORK_URL = "https://cloud.lin-q.ru/s/bEqr922zr6NXHy4/download"

# Пояснения:
#  ONNX       — модель после tools/export.py. Если её нет, возьмём models/yolo11n.onnx: прогон всего маршрута на стандартной модели.
#  CALIB_DIR  — кадры для калибровки (путь внутри проекта или на Диске). Должны быть похожи на реальные кадры с камер.
#  PRESET     — o1 надёжно; o5 быстрее, но, по документации SDK, может не скомпилироваться (тогда вернуться к o1).
#  PERCENTILE — 0 = как в SDK по умолчанию; 99.9 — устойчивее к выбросам на больших калибровочных наборах.
#  N_CHECK    — на скольких кадрах сравнить INT8 с исходной моделью; 0 — не проверять.

In [ ]:
#@title 2. Проект safety_monitor
import os, glob, zipfile, shutil

os.chdir("/content")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    project_zip = os.path.join(DRIVE_DIR, "safety_monitor.zip")
else:
    zips = sorted(glob.glob("/content/safety_monitor*.zip"))
    if not zips:
        from google.colab import files
        print("Загрузи safety_monitor.zip")
        uploaded = files.upload()
        zips = [os.path.join("/content", n) for n in uploaded if n.endswith(".zip")]
    project_zip = zips[0]

shutil.rmtree("/content/safety_monitor", ignore_errors=True)
zipfile.ZipFile(project_zip).extractall("/content")
os.chdir("/content/safety_monitor")

if not os.path.exists(ONNX):
    print(f"{ONNX} не найден — для проверки маршрута беру models/yolo11n.onnx")
    ONNX = "models/yolo11n.onnx"
print("Проект:", os.getcwd(), "| модель:", ONNX)

In [ ]:
#@title 3. Кадры для калибровки
import os, glob, zipfile, urllib.request

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
def count_images(d):
    return sum(1 for p in glob.glob(os.path.join(d, "*")) if p.lower().endswith(IMG_EXT)) if os.path.isdir(d) else 0

if count_images(CALIB_DIR) == 0 and USE_DRIVE and count_images(os.path.join(DRIVE_DIR, "calib")) > 0:
    CALIB_DIR = os.path.join(DRIVE_DIR, "calib")

if count_images(CALIB_DIR) == 0:
    if "yolo11n" in ONNX:
        # стандартная модель COCO -> калибруем на 128 картинках COCO (официальный архив ultralytics)
        urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip",
                                   "/content/coco128.zip")
        zipfile.ZipFile("/content/coco128.zip").extractall("/content")
        CALIB_DIR = "/content/coco128/images/train2017"
    else:
        from google.colab import files
        print("Загрузи zip с кадрами для калибровки (val датасета или стоп-кадры с камер, 100–300 шт.)")
        up = files.upload()
        name = [n for n in up if n.endswith(".zip")][0]
        zipfile.ZipFile(name).extractall("/content/calib_upload")
        dirs = sorted({os.path.dirname(p) for p in glob.glob("/content/calib_upload/**/*", recursive=True)
                       if p.lower().endswith(IMG_EXT)}, key=count_images, reverse=True)
        CALIB_DIR = dirs[0]

print(f"Калибровка: {CALIB_DIR} ({count_images(CALIB_DIR)} кадров, возьмём до {N_CALIB})")

In [ ]:
#@title 4. Скачать SDK LinQ (~500 МБ)
import os, zipfile, glob, shutil

if not os.path.isdir("/content/tpu_framework"):
    !wget -q --show-progress -O /content/Framework.zip "{FRAMEWORK_URL}"
    outer = zipfile.ZipFile("/content/Framework.zip")          # упадёт здесь, если архив скачался не полностью
    inner = [n for n in outer.namelist() if n.endswith(".zip")][0]
    outer.extract(inner, "/content/fw")
    zipfile.ZipFile(os.path.join("/content/fw", inner)).extractall("/content")
print("\n".join(sorted(os.listdir("/content/tpu_framework/packets"))))

In [ ]:
#@title 5. Окружение Python 3.12 с SDK (~10 мин)
PY = "/content/venv312/bin/python"
!pip -q install uv
!uv venv /content/venv312 --python 3.12 --seed -q
# torch 2.9.1 — версия, которую требует SDK; берём сборку для CPU (меньше и быстрее ставится)
!uv pip install -q --python {PY} torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cpu
# пакеты SDK: requirements.txt ссылается на .whl из папки packets, поэтому ставим из неё
!cd /content/tpu_framework/packets && uv pip install -q --python {PY} --find-links . -r requirements.txt
# нашему коду нужен ещё OpenCV (версия, совместимая с numpy 1.26 из SDK)
!uv pip install -q --python {PY} "opencv-python-headless==4.10.0.84" pyyaml
!{PY} -c "import sys, numpy, torch, onnx, tpu_framework; print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| torch', torch.__version__, '| onnx', onnx.__version__, '| tpu_framework OK')"

In [ ]:
#@title 6. Разрез модели + калибровка INT8 + компиляция в .tpu
pct = f"--percentile {PERCENTILE}" if PERCENTILE else ""
!{PY} tools/compile_linq.py compile --onnx "{ONNX}" --calib-dir "{CALIB_DIR}" --calib {N_CALIB} --preset {PRESET} --check {N_CHECK} {pct}

**Как читать вывод ячейки 6**
- `Проверка: голова+хвост ... OK` — разрез корректен, части вместе дают тот же выход, что исходная модель.
- `Готово: ..._head.tpu` — программа для TPU собрана.
- `INT8 на CPU против FP32: recall …, precision …` — сколько детекций исходной модели сохранилось после квантования.
  Хорошо — выше 0.9. Если сильно ниже: увеличь `N_CALIB`, поставь `PERCENTILE = 99.9`, проверь, что кадры калибровки похожи на реальные.

**Если компиляция упала**
- с `PRESET = "o5"` — вернись на `o1` (так советует документация SDK);
- на блоке внимания (MatMul / Softmax в `model.10`) — обучи модель на YOLOv8n (`python tools/train.py --model yolov8n.pt …`):
  там внимания нет, а `yolov8n_pose` есть в списке поддерживаемых моделей LinQ.

In [ ]:
#@title 7. Скачать результат
import os, glob, zipfile, shutil

stem = os.path.splitext(os.path.basename(ONNX))[0]
outs = [f"models/{stem}_head.tpu", f"models/{stem}_head.onnx", f"models/{stem}_tail.onnx",
        f"models/{stem}_head_build/thresholds.json"]
outs = [p for p in outs if os.path.exists(p)]
result = f"/content/{stem}_linq.zip"
with zipfile.ZipFile(result, "w") as z:
    for p in outs:
        z.write(p, os.path.basename(p) if p.endswith(".json") else p)
print("В архиве:", outs)

if USE_DRIVE:
    shutil.copy(result, DRIVE_DIR)
    print("Скопировано на Диск:", DRIVE_DIR)
from google.colab import files
files.download(result)

## Дальше — на стенде LinQ

1. VPN (`Hakathon.ovpn`, логин и пароль у капитана) → Jupyter `http://hackathon.lin-q.ru/`.
2. Загрузить проект и файлы из архива в `models/`; программу `.tpu` скопировать в `/auto/cache/<ЛОГИН>/`.
3. В `config.yaml`:
   ```yaml
   model:
     backend: linq
   linq:
     model_path: /auto/cache/<ЛОГИН>/ppe_head.tpu
     head_onnx: models/ppe_head.onnx
     tail_onnx: models/ppe_tail.onnx
     names_from: models/ppe.onnx
   ```
4. Сверка с эталоном и замер скорости:
   ```bash
   python tools/compare.py --images <кадры> --ref onnx:models/ppe.onnx --test linq
   python tools/benchmark.py --image data/sample.jpg --backend onnx:models/ppe.onnx --backend linq
   ```